<a href="https://colab.research.google.com/github/amitpoa/AI-Agent-by-Amit/blob/main/AI_Agent_for_Clash_of_Clans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 11.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


In [ ]:
pip install -U "langchain[google-genai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.9
    Uninstalling langgraph-1.1.9:
      Successfully uninstalled langgraph-1.1.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.15
    Uninstalling langchain-1.2.15:
      Successfully uninstalled langchain-1.2.15


In [ ]:
pip install -U "langchain-openrouter"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 9.1 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

# Set the environment variable for your key
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Set the environment variable for your key
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

In [ ]:
import math
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent
from dotenv import load_dotenv

In [ ]:
clans_status = {
                "resource": {"gold": 5000, "elixir": 1500},
                "topps":   {"barbarians": 5, "archers": 6},
                "clan":    {"town_hall_level": 5, "wall_level": 4}
 }


In [ ]:
@tool

def get_clan_report() -> str:
  """
  Returns the current resources and army count of the village.
  """
  print(clans_status)
  return str(clans_status)

In [ ]:
@tool

def train_troops(unit_type: str, quantity: int) -> str:
    """
    Trains troops.
    Costs: 'barbarian' = 50 gold, 'archer' = 25 elixir.
    """
    unit_type = unit_type.lower()

    if unit_type == "barbarian":
        cost_per_unit = 50
        res_type = "gold"
        troop_key = "barbarians"
    elif unit_type == "archer":
        cost_per_unit = 25
        res_type = "elixir"
        troop_key = "archers"
    else:
        return "Error: Unknown unit type."

    total_cost = cost_per_unit * quantity

    # Accessing nested dictionary
    if clans_status["resource"][res_type] >= total_cost:
        clans_status["resource"][res_type] -= total_cost
        clans_status["troops"][troop_key] += quantity
        msg = f"SUCCESS: Trained {quantity} {unit_type}s. Cost: {total_cost} {res_type}."
        print(f"\n[TOOL CALL] train_troops: {msg}")
        return msg
    else:
        msg = f"FAILURE: Need {total_cost} {res_type}, but only have {clans_status['resource'][res_type]}."
        print(f"\n[TOOL CALL] train_troops: {msg}")
        return msg

In [ ]:
@tool
def collect_resources() -> str:
    """Collects 500 Gold and 500 Elixir from the mines and update my resource count"""
    clans_status["resource"]["gold"] += 500
    clans_status["resource"]["elixir"] += 500
    print(f"\n[TOOL CALL] collect_resources: Added 500 to both pools.")
    return "Mines emptied! Added 500 Gold and 500 Elixir."

In [ ]:
model = ChatOpenRouter(model="qwen/qwen-2.5-coder-32b-instruct", temperature=0.2)
tools = [get_clan_report, train_troops, collect_resources]
agent = create_agent(model, tools, system_prompt="You are a Strategic Clan Chief managing a village. Use tools to monitor resources, train troops, and upgrade buildings. You must always verify the current village state before narrating actions and never describe having resources or troops that the tools have not confirmed.")

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.2)
tools = [get_clan_report, train_troops, collect_resources]
agent = create_agent(model, tools, system_prompt="You are a Strategic Clan Chief managing a village. Use tools to monitor resources, train troops, and upgrade buildings. You must always verify the current village state before narrating actions and never describe having resources or troops that the tools have not confirmed.")

In [ ]:
print("---Clash of Clans ready")
print(f"Starting State: {clans_status}")

---Clash of Clans ready
Starting State: {'resource': {'gold': 2750, 'elixir': 1500}, 'topps': {'barbarians': 5, 'archers': 6}, 'clan': {'town_hall_level': 5, 'wall_level': 4}}


In [ ]:
while True:
    user_input = input("\nWhat would you like to do? (or 'quit'): ")
    if user_input.lower() in ["quit", "exit"]:
        break

    # The agent decides which tool to call based on the natural language input
    response = agent.invoke({"messages": [("user", user_input)]})

    print(f"AI: {response['messages'][-1].content[0]['text']}")
    print(f"(Game Status: {clans_status})")

In [ ]:
while True:
    user_input = input("\nWhat would you like to do? (or 'quit'): ")
    if user_input.lower() in ["quit", "exit"]:
        break

    # The agent decides which tool to call based on the natural language input
    response = agent.invoke({"messages": [("user", user_input)]})

    print(f"AI: {response['messages'][-1].content}")
    print(f"(Game Status: {clans_status})")


What would you like to do? (or 'quit'): train my troops
{'resource': {'gold': 2950, 'elixir': 2000}, 'topps': {'barbarians': 5, 'archers': 6}, 'clan': {'town_hall_level': 5, 'wall_level': 4}}
AI: 
I can see your village has:
- **Gold:** 2,950
- **Elixir:** 2,000
- **Barbarians:** 5
- **Archers:** 6
- **Town Hall Level:** 5

What type of troops would you like to train and how many? I can train:
- **Barbarians** (costs 50 gold each)
- **Archers** (costs 25 elixir each)

(Game Status: {'resource': {'gold': 2950, 'elixir': 2000}, 'topps': {'barbarians': 5, 'archers': 6}, 'clan': {'town_hall_level': 5, 'wall_level': 4}})

What would you like to do? (or 'quit'): quit
